# DATATHON 2026 - Exploratory Data Analysis (EDA)
## Master Table Visualization Dashboard
---

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Style
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

print('Libraries loaded.')

Libraries loaded.


In [2]:
# Load data
df = pd.read_csv(r'd:\DATATHON-2026-GenApLucUWU\data\output\master_table.csv')

# Parse dates
date_cols = ['order_date', 'signup_date', 'ship_date', 'delivery_date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Derived columns
df['lead_time'] = (df['delivery_date'] - df['order_date']).dt.days
df['order_year'] = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.to_period('M')
df['order_year_month'] = df['order_date'].dt.to_period('M').astype(str)
df['gross_profit'] = df['total_item_revenue'] - df['cogs'] * df['total_quantity']
df['net_revenue'] = df['total_item_revenue'] - df['total_refund_amount'].fillna(0)

print(f'Shape: {df.shape}')
df.dtypes

FileNotFoundError: [Errno 2] No such file or directory: 'd:\\DATATHON-2026-GenApLucUWU\\data\\output\\master_table.csv'

---
# NHOM 1: Kham pha Phan phoi Du lieu (Distribution)
---

In [ ]:
# 1.1 Histogram - Phan phoi cua Price, Total Payment, Shipping Fee
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col, title in zip(axes, ['price', 'total_payment', 'shipping_fee'],
                           ['Price', 'Total Payment', 'Shipping Fee']):
    data = df[col].dropna()
    ax.hist(data, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(f'Histogram: {title}', fontweight='bold')
    ax.set_xlabel(title)
    ax.set_ylabel('Frequency')
    ax.axvline(data.median(), color='red', linestyle='--', label=f'Median={data.median():,.0f}')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 1.2 KDE Plot - So sanh Lead Time giua cac Region (city groups)
top_cities = df['city'].value_counts().head(5).index.tolist()
fig, ax = plt.subplots(figsize=(10, 5))

for city in top_cities:
    subset = df[df['city'] == city]['lead_time'].dropna()
    if len(subset) > 10:
        subset.plot.kde(ax=ax, label=city, linewidth=2)

ax.set_title('KDE: Lead Time theo Top 5 Thanh pho', fontweight='bold')
ax.set_xlabel('Lead Time (days)')
ax.set_xlim(0, df['lead_time'].quantile(0.99))
ax.legend(title='City')
plt.tight_layout()
plt.show()

In [ ]:
# 1.3 Box Plot - Payment Value & Lead Time theo Order Status
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(data=df, x='order_status', y='total_payment', ax=axes[0],
            showfliers=True, flierprops=dict(marker='o', markersize=2, alpha=0.3))
axes[0].set_title('Box Plot: Total Payment theo Order Status', fontweight='bold')
axes[0].set_ylim(0, df['total_payment'].quantile(0.95))

sns.boxplot(data=df, x='order_status', y='lead_time', ax=axes[1],
            showfliers=True, flierprops=dict(marker='o', markersize=2, alpha=0.3))
axes[1].set_title('Box Plot: Lead Time theo Order Status', fontweight='bold')
axes[1].set_ylim(0, df['lead_time'].quantile(0.95))

plt.tight_layout()
plt.show()

In [ ]:
# 1.4 Violin Plot - Phan phoi Rating theo Order Source
df_rating = df.dropna(subset=['avg_rating'])

fig, ax = plt.subplots(figsize=(12, 6))
sns.violinplot(data=df_rating, x='order_source', y='avg_rating', ax=ax,
               inner='box', cut=0, palette='Set2')
ax.set_title('Violin Plot: Diem Rating theo Order Source', fontweight='bold')
ax.set_xlabel('Order Source')
ax.set_ylabel('Average Rating')
plt.tight_layout()
plt.show()

In [ ]:
# 1.5 ECDF Plot - Total Payment
fig, ax = plt.subplots(figsize=(10, 5))
payment_sorted = np.sort(df['total_payment'].dropna())
ecdf_y = np.arange(1, len(payment_sorted) + 1) / len(payment_sorted)
ax.plot(payment_sorted, ecdf_y, linewidth=1.5, color='teal')
ax.set_title('ECDF: Total Payment', fontweight='bold')
ax.set_xlabel('Total Payment')
ax.set_ylabel('Cumulative Proportion')
ax.axhline(0.8, color='red', linestyle='--', alpha=0.6, label='80%')
ax.axhline(0.5, color='orange', linestyle='--', alpha=0.6, label='50%')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
# NHOM 2: So sanh & Xep hang (Comparison & Categorization)
---

In [ ]:
# 2.1 Bar Chart - Tong Doanh thu theo Category
rev_by_cat = df.groupby('category')['total_item_revenue'].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
rev_by_cat.plot.barh(ax=ax, color=sns.color_palette('viridis', len(rev_by_cat)), edgecolor='white')
ax.set_title('Tong Doanh thu theo Category', fontweight='bold')
ax.set_xlabel('Total Revenue')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:.1f}B'))
for i, v in enumerate(rev_by_cat):
    ax.text(v + rev_by_cat.max() * 0.01, i, f'{v/1e9:.2f}B', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# 2.1b Bar Chart - So luong khach hang theo Age Group
cust_by_age = df.groupby('age_group')['customer_id'].nunique().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
cust_by_age.plot.bar(ax=ax, color=sns.color_palette('coolwarm', len(cust_by_age)), edgecolor='white')
ax.set_title('So luong Khach hang theo Age Group', fontweight='bold')
ax.set_ylabel('Unique Customers')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for i, v in enumerate(cust_by_age):
    ax.text(i, v + cust_by_age.max() * 0.01, f'{v:,}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# 2.2 Lollipop Chart - Doanh thu theo Top 30 City
rev_by_city = df.groupby('city')['total_item_revenue'].sum().sort_values(ascending=False).head(30)
rev_by_city = rev_by_city.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 10))
ax.hlines(y=rev_by_city.index, xmin=0, xmax=rev_by_city.values, color='steelblue', linewidth=1.5)
ax.scatter(rev_by_city.values, rev_by_city.index, color='steelblue', s=50, zorder=3)
ax.set_title('Lollipop: Doanh thu Top 30 Thanh pho', fontweight='bold')
ax.set_xlabel('Total Revenue')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:.1f}B'))
plt.tight_layout()
plt.show()

In [ ]:
# 2.3 Pareto Chart - Top San pham tao ra 80% Doanh thu
rev_by_prod = df.groupby('product_name')['total_item_revenue'].sum().sort_values(ascending=False)
cumulative_pct = rev_by_prod.cumsum() / rev_by_prod.sum() * 100
top_n = (cumulative_pct <= 85).sum() + 1
top_n = min(top_n, 30)  # Cap at 30 for readability

pareto_rev = rev_by_prod.head(top_n)
pareto_cum = cumulative_pct.head(top_n)

fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.bar(range(len(pareto_rev)), pareto_rev.values, color='steelblue', alpha=0.8)
ax1.set_ylabel('Revenue', color='steelblue')
ax1.set_xticks(range(len(pareto_rev)))
ax1.set_xticklabels(pareto_rev.index, rotation=60, ha='right', fontsize=7)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:.1f}B'))

ax2 = ax1.twinx()
ax2.plot(range(len(pareto_cum)), pareto_cum.values, color='red', marker='o', markersize=4, linewidth=2)
ax2.axhline(80, color='red', linestyle='--', alpha=0.5, label='80%')
ax2.set_ylabel('Cumulative %', color='red')
ax2.set_ylim(0, 105)
ax2.legend(loc='center right')

ax1.set_title('Pareto Chart: San pham theo Doanh thu (Quy tac 80/20)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 2.4 Radar Chart - Danh gia "suc khoe" cac Segment
segment_stats = df.groupby('segment').agg(
    avg_revenue=('total_item_revenue', 'mean'),
    avg_quantity=('total_quantity', 'mean'),
    avg_rating=('avg_rating', 'mean'),
    return_rate=('total_return_qty', lambda x: x.notna().mean()),
    avg_lead_time=('lead_time', 'mean')
).dropna()

# Normalize 0-1
seg_norm = (segment_stats - segment_stats.min()) / (segment_stats.max() - segment_stats.min() + 1e-9)

categories_radar = seg_norm.columns.tolist()
N = len(categories_radar)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors = sns.color_palette('husl', len(seg_norm))

for idx, (seg_name, row) in enumerate(seg_norm.iterrows()):
    values = row.values.tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=2, label=seg_name, color=colors[idx])
    ax.fill(angles, values, alpha=0.1, color=colors[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories_radar, fontsize=9)
ax.set_title('Radar Chart: Danh gia Segment', fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=8)
plt.tight_layout()
plt.show()

---
# NHOM 3: Kham pha Cau truc & Ty trong (Composition / Part-to-Whole)
---

In [ ]:
# 3.1 Donut Chart - Ty le Phuong thuc thanh toan & Gioi tinh
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Payment method
pay_counts = df['payment_method'].value_counts()
colors_pay = sns.color_palette('Set2', len(pay_counts))
wedges, texts, autotexts = axes[0].pie(pay_counts, labels=pay_counts.index, autopct='%1.1f%%',
                                        colors=colors_pay, pctdistance=0.82, startangle=90)
centre_circle = plt.Circle((0, 0), 0.60, fc='white')
axes[0].add_artist(centre_circle)
axes[0].set_title('Donut: Phuong thuc Thanh toan', fontweight='bold')

# Gender
gender_counts = df['gender'].value_counts()
colors_gen = sns.color_palette('Pastel1', len(gender_counts))
wedges2, texts2, autotexts2 = axes[1].pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%',
                                           colors=colors_gen, pctdistance=0.82, startangle=90)
centre_circle2 = plt.Circle((0, 0), 0.60, fc='white')
axes[1].add_artist(centre_circle2)
axes[1].set_title('Donut: Gioi tinh Khach hang', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 3.2 Stacked Bar Chart (100%) - Ty le Age Group trong moi Order Source
cross = pd.crosstab(df['order_source'], df['age_group'], normalize='index') * 100

fig, ax = plt.subplots(figsize=(12, 6))
cross.plot.bar(stacked=True, ax=ax, colormap='tab20', edgecolor='white', linewidth=0.5)
ax.set_title('Stacked Bar 100%: Ty le Age Group trong moi Order Source', fontweight='bold')
ax.set_ylabel('Percentage (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Age Group', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# 3.3 Treemap - He thong Danh muc San pham (Category > Segment > Top Products)
tree_data = df.groupby(['category', 'segment', 'product_name'])['total_item_revenue'].sum().reset_index()
# Keep top products per segment for readability
tree_data = tree_data.sort_values('total_item_revenue', ascending=False)
tree_top = tree_data.groupby(['category', 'segment']).head(3).reset_index(drop=True)

fig = px.treemap(tree_top, path=['category', 'segment', 'product_name'],
                 values='total_item_revenue',
                 color='total_item_revenue',
                 color_continuous_scale='Blues',
                 title='Treemap: Cau truc Danh muc San pham theo Doanh thu')
fig.update_layout(height=600)
fig.show()

In [ ]:
# 3.4 Waterfall Chart - Tu Gross Revenue den Net Profit
total_gross = df['total_item_revenue'].sum()
total_discount = df['total_refund_amount'].fillna(0).sum()
total_cogs = (df['cogs'] * df['total_quantity']).sum()
total_shipping = df['shipping_fee'].fillna(0).sum()
net_profit = total_gross - total_discount - total_cogs - total_shipping

fig = go.Figure(go.Waterfall(
    name='', orientation='v',
    measure=['absolute', 'relative', 'relative', 'relative', 'total'],
    x=['Gross Revenue', 'Refund/Returns', 'COGS', 'Shipping Fee', 'Net Profit'],
    y=[total_gross, -total_discount, -total_cogs, -total_shipping, 0],
    text=[f'{total_gross/1e9:.1f}B', f'-{total_discount/1e9:.1f}B',
          f'-{total_cogs/1e9:.1f}B', f'-{total_shipping/1e9:.1f}B',
          f'{net_profit/1e9:.1f}B'],
    textposition='outside',
    connector={'line': {'color': 'rgb(63, 63, 63)'}},
    increasing={'marker': {'color': '#2ecc71'}},
    decreasing={'marker': {'color': '#e74c3c'}},
    totals={'marker': {'color': '#3498db'}}
))
fig.update_layout(title='Waterfall Chart: Tu Gross Revenue den Net Profit',
                  height=500, showlegend=False)
fig.show()

---
# NHOM 4: Moi quan he & Tuong quan (Relationship & Correlation)
---

In [ ]:
# 4.1 Scatter Plot - Price vs Quantity (Do co gian cua Cau theo Gia)
sample = df[['price', 'total_quantity', 'category']].dropna().sample(n=min(5000, len(df)), random_state=42)

fig, ax = plt.subplots(figsize=(10, 6))
categories = sample['category'].unique()
colors = sns.color_palette('husl', len(categories))

for cat, color in zip(categories, colors):
    mask = sample['category'] == cat
    ax.scatter(sample.loc[mask, 'price'], sample.loc[mask, 'total_quantity'],
               alpha=0.4, s=20, label=cat, color=color)

ax.set_title('Scatter: Price vs Quantity (Do co gian Cau theo Gia)', fontweight='bold')
ax.set_xlabel('Unit Price')
ax.set_ylabel('Total Quantity')
ax.legend(title='Category', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# 4.2 Bubble Chart - Revenue vs Lead Time, size=stock_on_hand, color=order_source
bubble_data = df[['total_item_revenue', 'lead_time', 'stock_on_hand', 'order_source']].dropna()
bubble_sample = bubble_data.sample(n=min(3000, len(bubble_data)), random_state=42)

fig = px.scatter(bubble_sample, x='lead_time', y='total_item_revenue',
                 size='stock_on_hand', color='order_source',
                 size_max=30, opacity=0.5,
                 title='Bubble Chart: Revenue vs Lead Time (size=Stock, color=Order Source)',
                 labels={'lead_time': 'Lead Time (days)', 'total_item_revenue': 'Revenue'})
fig.update_layout(height=550)
fig.show()

In [ ]:
# 4.3 Hexbin Plot - Order Value vs Lead Time
hex_data = df[['total_payment', 'lead_time']].dropna()

fig, ax = plt.subplots(figsize=(10, 6))
hb = ax.hexbin(hex_data['lead_time'], hex_data['total_payment'],
               gridsize=40, cmap='YlOrRd', mincnt=1)
ax.set_title('Hexbin: Lead Time vs Total Payment', fontweight='bold')
ax.set_xlabel('Lead Time (days)')
ax.set_ylabel('Total Payment')
ax.set_ylim(0, hex_data['total_payment'].quantile(0.95))
plt.colorbar(hb, ax=ax, label='Count')
plt.tight_layout()
plt.show()

In [ ]:
# 4.4 Correlation Heatmap
num_cols = ['price', 'cogs', 'total_payment', 'total_item_revenue', 'total_quantity',
            'shipping_fee', 'avg_rating', 'lead_time', 'stock_on_hand',
            'fill_rate', 'sell_through_rate', 'installments']
corr_matrix = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Heatmap: Ma tran Tuong quan', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 4.5 Pairplot (subset for performance)
pair_cols = ['price', 'total_quantity', 'avg_rating', 'lead_time', 'shipping_fee']
pair_data = df[pair_cols + ['order_status']].dropna().sample(n=min(2000, len(df)), random_state=42)

g = sns.pairplot(pair_data, hue='order_status', palette='Set2',
                 plot_kws={'alpha': 0.4, 's': 15}, diag_kind='kde', height=2.2)
g.figure.suptitle('Pairplot: Cac bien so chinh theo Order Status', y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

---
# NHOM 5: Xu huong & Thoi gian (Time-Series & Trend)
---

In [ ]:
# 5.1 Line Chart - Revenue theo Thang
monthly_rev = df.groupby(df['order_date'].dt.to_period('M')).agg(
    revenue=('total_item_revenue', 'sum'),
    orders=('order_id', 'nunique')
).reset_index()
monthly_rev['order_date'] = monthly_rev['order_date'].dt.to_timestamp()

fig, ax1 = plt.subplots(figsize=(16, 6))
ax1.plot(monthly_rev['order_date'], monthly_rev['revenue'], color='steelblue',
         linewidth=2, marker='o', markersize=3, label='Revenue')
ax1.set_ylabel('Revenue', color='steelblue')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:.1f}B'))

ax2 = ax1.twinx()
ax2.plot(monthly_rev['order_date'], monthly_rev['orders'], color='coral',
         linewidth=2, linestyle='--', marker='s', markersize=3, label='Orders')
ax2.set_ylabel('Order Count', color='coral')

ax1.set_title('Line Chart: Revenue & Order Count theo Thang', fontweight='bold')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# 5.2 Stacked Area Chart - Doanh thu theo City (Top 5) qua thoi gian
top5_cities = df.groupby('city')['total_item_revenue'].sum().nlargest(5).index
area_data = df[df['city'].isin(top5_cities)].groupby(
    [df['order_date'].dt.to_period('M'), 'city']
)['total_item_revenue'].sum().reset_index()
area_data['order_date'] = area_data['order_date'].dt.to_timestamp()
area_pivot = area_data.pivot_table(index='order_date', columns='city',
                                    values='total_item_revenue', fill_value=0)

fig, ax = plt.subplots(figsize=(16, 6))
area_pivot.plot.area(ax=ax, alpha=0.7, linewidth=1)
ax.set_title('Stacked Area: Doanh thu Top 5 Thanh pho theo Thang', fontweight='bold')
ax.set_ylabel('Revenue')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:.1f}B'))
ax.legend(title='City', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# 5.3 Time-Series Decomposition - Doanh thu hang thang
from statsmodels.tsa.seasonal import seasonal_decompose

ts_data = df.groupby(df['order_date'].dt.to_period('M'))['total_item_revenue'].sum()
ts_data.index = ts_data.index.to_timestamp()
ts_data = ts_data.sort_index()

# Need at least 2 full cycles; try period=12 (monthly seasonality)
if len(ts_data) >= 24:
    decomposition = seasonal_decompose(ts_data, model='additive', period=12)
    fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
    
    decomposition.observed.plot(ax=axes[0], color='steelblue')
    axes[0].set_ylabel('Observed')
    axes[0].set_title('Time-Series Decomposition: Doanh thu hang thang', fontweight='bold')
    
    decomposition.trend.plot(ax=axes[1], color='coral')
    axes[1].set_ylabel('Trend')
    
    decomposition.seasonal.plot(ax=axes[2], color='green')
    axes[2].set_ylabel('Seasonality')
    
    decomposition.resid.plot(ax=axes[3], color='gray')
    axes[3].set_ylabel('Residuals')
    
    plt.tight_layout()
    plt.show()
else:
    print(f'Not enough data for decomposition (only {len(ts_data)} periods, need >= 24)')

---
# NHOM 6: Kham pha Khong gian & Dia ly (Geospatial)
---

In [ ]:
# 6.1 Choropleth-style Heatmap - Doanh thu & Ty le Cancel theo City
# (Without geojson, we use a ranked bar heatmap as geographic proxy)
geo_data = df.groupby('city').agg(
    revenue=('total_item_revenue', 'sum'),
    total_orders=('order_id', 'nunique'),
    cancelled=('order_status', lambda x: (x == 'cancelled').sum())
).reset_index()
geo_data['cancel_rate'] = geo_data['cancelled'] / geo_data['total_orders'] * 100
geo_top = geo_data.nlargest(30, 'revenue')

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Revenue heatmap bar
geo_rev = geo_top.sort_values('revenue', ascending=True)
colors_rev = plt.cm.Reds(geo_rev['revenue'] / geo_rev['revenue'].max())
axes[0].barh(geo_rev['city'], geo_rev['revenue'], color=colors_rev)
axes[0].set_title('Top 30 City: Doanh thu (mau do = cao)', fontweight='bold')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:.1f}B'))

# Cancel rate heatmap bar
geo_cancel = geo_top.sort_values('cancel_rate', ascending=True)
colors_cancel = plt.cm.Greys(geo_cancel['cancel_rate'] / geo_cancel['cancel_rate'].max())
axes[1].barh(geo_cancel['city'], geo_cancel['cancel_rate'], color=colors_cancel)
axes[1].set_title('Top 30 City: Ty le Cancel % (mau xam = cao)', fontweight='bold')
axes[1].set_xlabel('Cancel Rate %')

plt.tight_layout()
plt.show()

In [ ]:
# 6.2 Bubble Map (simulated) - Doanh thu va So don theo City (top 20)
geo_bubble = geo_data.nlargest(20, 'revenue')

fig = px.scatter(geo_bubble, x='city', y='revenue', size='total_orders',
                 color='cancel_rate', color_continuous_scale='RdYlGn_r',
                 size_max=40,
                 title='Bubble: Doanh thu, So don, Ty le Cancel theo City',
                 labels={'revenue': 'Revenue', 'total_orders': 'Orders', 'cancel_rate': 'Cancel %'})
fig.update_layout(height=500, xaxis_tickangle=-45)
fig.show()

---
# NHOM 7: Bieu do Dac thu Chuyen sau Kinh doanh (Advanced Business)
---

In [ ]:
# 7.1 Cohort/Retention Heatmap
df_cohort = df[['customer_id', 'order_date']].dropna().copy()
df_cohort['order_month'] = df_cohort['order_date'].dt.to_period('M')

# First purchase month per customer
df_cohort['cohort_month'] = df_cohort.groupby('customer_id')['order_month'].transform('min')
df_cohort['month_offset'] = (df_cohort['order_month'].dt.year - df_cohort['cohort_month'].dt.year) * 12 + \
                             (df_cohort['order_month'].dt.month - df_cohort['cohort_month'].dt.month)

# Build cohort table
cohort_table = df_cohort.groupby(['cohort_month', 'month_offset'])['customer_id'].nunique().reset_index()
cohort_pivot = cohort_table.pivot_table(index='cohort_month', columns='month_offset',
                                         values='customer_id')

# Retention rate
cohort_size = cohort_pivot.iloc[:, 0]
retention = cohort_pivot.divide(cohort_size, axis=0) * 100

# Show last 12 cohorts, first 12 months
retention_display = retention.iloc[-12:, :13]

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(retention_display, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, vmin=0, vmax=100,
            cbar_kws={'label': 'Retention %'})
ax.set_title('Cohort Retention Heatmap (Last 12 Cohorts)', fontweight='bold')
ax.set_xlabel('Month Offset')
ax.set_ylabel('Cohort Month')
plt.tight_layout()
plt.show()

In [ ]:
# 7.2 Sankey Diagram - Hanh trinh Don hang
total_orders = df['order_id'].nunique()

# Status counts
status_counts = df.groupby('order_status')['order_id'].nunique()
delivered = status_counts.get('delivered', 0)
shipped = status_counts.get('shipped', 0)
cancelled = status_counts.get('cancelled', 0)
returned = status_counts.get('returned', 0)

# Sankey: Total -> Cancelled / Processing
# Processing -> Delivered / Shipped (in transit) / Returned
processing = total_orders - cancelled
successful_delivery = delivered

labels = ['Total Orders', 'Cancelled', 'Processing', 'Delivered', 'Shipped (In Transit)', 'Returned']
source = [0, 0, 2, 2, 2]
target = [1, 2, 3, 4, 5]
values = [cancelled, processing, successful_delivery, shipped, returned]

fig = go.Figure(data=[go.Sankey(
    node=dict(pad=15, thickness=20, line=dict(color='black', width=0.5),
              label=labels,
              color=['#3498db', '#e74c3c', '#f39c12', '#2ecc71', '#9b59b6', '#e67e22']),
    link=dict(source=source, target=target, value=values,
              color=['rgba(231,76,60,0.4)', 'rgba(243,156,18,0.4)',
                     'rgba(46,204,113,0.4)', 'rgba(155,89,182,0.4)',
                     'rgba(230,126,34,0.4)'])
)])
fig.update_layout(title_text='Sankey: Hanh trinh Don hang (Order Flow)',
                  font_size=12, height=500)
fig.show()

In [ ]:
# 7.3 Funnel Chart - Conversion Pipeline
total = df['order_id'].nunique()
paid = df[df['order_status'] != 'cancelled']['order_id'].nunique()
shipped_delivered = df[df['order_status'].isin(['shipped', 'delivered', 'returned'])]['order_id'].nunique()
delivered_count = df[df['order_status'] == 'delivered']['order_id'].nunique()
reviewed = df[df['avg_rating'].notna()]['order_id'].nunique()

funnel_stages = ['Total Orders', 'Paid (Not Cancelled)', 'Shipped/Delivered', 'Delivered', 'Reviewed']
funnel_values = [total, paid, shipped_delivered, delivered_count, reviewed]

fig = go.Figure(go.Funnel(
    y=funnel_stages, x=funnel_values,
    textinfo='value+percent initial',
    marker=dict(color=['#3498db', '#2ecc71', '#f39c12', '#e74c3c', '#9b59b6']),
    connector={'line': {'color': 'royalblue', 'width': 2}}
))
fig.update_layout(title='Funnel Chart: Order Conversion Pipeline',
                  height=500)
fig.show()

In [ ]:
# 7.4 Cohort by Acquisition Channel - So sanh Retention
df_acq = df[['customer_id', 'order_date', 'acquisition_channel']].dropna().copy()
df_acq['order_month'] = df_acq['order_date'].dt.to_period('M')

# First purchase per customer
first_purchase = df_acq.groupby('customer_id').agg(
    cohort_month=('order_month', 'min'),
    channel=('acquisition_channel', 'first')
).reset_index()

df_acq = df_acq.merge(first_purchase, on='customer_id')
df_acq['month_offset'] = (df_acq['order_month'].dt.year - df_acq['cohort_month'].dt.year) * 12 + \
                           (df_acq['order_month'].dt.month - df_acq['cohort_month'].dt.month)

# Retention by channel for first 6 months
channel_retention = []
for ch in df_acq['channel'].unique():
    ch_data = df_acq[df_acq['channel'] == ch]
    cohort_sizes = ch_data[ch_data['month_offset'] == 0].groupby('cohort_month')['customer_id'].nunique()
    total_cohort = cohort_sizes.sum()
    for m in range(0, 7):
        active = ch_data[ch_data['month_offset'] == m]['customer_id'].nunique()
        channel_retention.append({'channel': ch, 'month': m, 'retention': active / total_cohort * 100})

ret_df = pd.DataFrame(channel_retention)

fig, ax = plt.subplots(figsize=(10, 6))
for ch in ret_df['channel'].unique():
    ch_ret = ret_df[ret_df['channel'] == ch]
    ax.plot(ch_ret['month'], ch_ret['retention'], marker='o', linewidth=2, label=ch)

ax.set_title('Retention theo Acquisition Channel (0-6 months)', fontweight='bold')
ax.set_xlabel('Month Offset')
ax.set_ylabel('Retention %')
ax.legend(title='Channel')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Summary
Notebook covers all 7 chart groups:
1. **Distribution**: Histogram, KDE, Box Plot, Violin Plot, ECDF
2. **Comparison**: Bar Chart, Lollipop, Pareto, Radar
3. **Composition**: Donut, Stacked Bar, Treemap, Waterfall
4. **Relationship**: Scatter, Bubble, Hexbin, Heatmap, Pairplot
5. **Time-Series**: Line, Area, Decomposition
6. **Geospatial**: City Revenue/Cancel heatmap, Bubble map
7. **Advanced Business**: Cohort Retention, Sankey, Funnel, Channel Retention

---
# PHAN TICH THEO GIAI DOAN (Period Analysis)
## Giai doan: 2012-2015, 2016-2019, 2020-2021, 2022
---

In [ ]:
# === SETUP: Dinh nghia giai doan va helper ===
def assign_period(year):
    if year <= 2015:
        return '2012-2015'
    elif year <= 2019:
        return '2016-2019'
    elif year <= 2021:
        return '2020-2021'
    else:
        return '2022'

df['period'] = df['order_year'].apply(assign_period)
df['month_num'] = df['order_date'].dt.month

period_order = ['2012-2015', '2016-2019', '2020-2021', '2022']
df['period'] = pd.Categorical(df['period'], categories=period_order, ordered=True)

print('Periods assigned.')
print(df.groupby('period')['order_id'].nunique())

## 8.1 TONG QUAN THEO GIAI DOAN (Overall by Period)
Cot chong 100%: Category, Size, Color, Segment — x-axis la 4 giai doan

In [ ]:
# 8.1a Stacked Bar 100%: Category theo Period
fig, axes = plt.subplots(2, 2, figsize=(20, 14))

# --- Category ---
cross_cat = pd.crosstab(df['period'], df['category'], normalize='index') * 100
cross_cat.plot.bar(stacked=True, ax=axes[0, 0], colormap='Set2', edgecolor='white', linewidth=0.5)
axes[0, 0].set_title('Loai quan ao ua chuong theo Giai doan', fontweight='bold')
axes[0, 0].set_ylabel('Percentage (%)')
axes[0, 0].set_xticklabels(axes[0, 0].get_xticklabels(), rotation=0)
axes[0, 0].legend(title='Category', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)

# --- Size ---
cross_size = pd.crosstab(df['period'], df['size'], normalize='index') * 100
size_order = ['XS', 'S', 'M', 'L', 'XL', 'XXL']
available_sizes = [s for s in size_order if s in cross_size.columns]
other_sizes = [s for s in cross_size.columns if s not in size_order]
cross_size = cross_size[available_sizes + other_sizes]
cross_size.plot.bar(stacked=True, ax=axes[0, 1], colormap='tab10', edgecolor='white', linewidth=0.5)
axes[0, 1].set_title('Size quan ao theo Giai doan', fontweight='bold')
axes[0, 1].set_ylabel('Percentage (%)')
axes[0, 1].set_xticklabels(axes[0, 1].get_xticklabels(), rotation=0)
axes[0, 1].legend(title='Size', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)

# --- Color ---
cross_color = pd.crosstab(df['period'], df['color'], normalize='index') * 100
cross_color.plot.bar(stacked=True, ax=axes[1, 0], colormap='tab20', edgecolor='white', linewidth=0.5)
axes[1, 0].set_title('Mau sac ua chuong theo Giai doan', fontweight='bold')
axes[1, 0].set_ylabel('Percentage (%)')
axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=0)
axes[1, 0].legend(title='Color', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)

# --- Segment ---
cross_seg = pd.crosstab(df['period'], df['segment'], normalize='index') * 100
cross_seg.plot.bar(stacked=True, ax=axes[1, 1], colormap='Pastel1', edgecolor='white', linewidth=0.5)
axes[1, 1].set_title('Phan khuc San pham (Segment) theo Giai doan', fontweight='bold')
axes[1, 1].set_ylabel('Percentage (%)')
axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=0)
axes[1, 1].legend(title='Segment', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)

plt.suptitle('TONG QUAN: Cot chong 100% theo 4 Giai doan', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 8.2 CHI TIET THEO THANG trong moi Giai doan
Moi giai doan: 4 bieu do cot chong (Category, Size, Color, Segment) — x-axis la thang 1-12

In [ ]:
# 8.2 Chi tiet theo Thang (1-12) cho moi Giai doan
# Helper: ve 4 stacked bar (category, size, color, segment) cho 1 period
def plot_period_monthly(df_period, period_name):
    fig, axes = plt.subplots(2, 2, figsize=(20, 12))
    month_labels = list(range(1, 13))
    
    # --- Category ---
    cross = pd.crosstab(df_period['month_num'], df_period['category'], normalize='index') * 100
    cross = cross.reindex(month_labels, fill_value=0)
    cross.plot.bar(stacked=True, ax=axes[0, 0], colormap='Set2', edgecolor='white', linewidth=0.5)
    axes[0, 0].set_title(f'Category theo Thang', fontweight='bold')
    axes[0, 0].set_ylabel('%')
    axes[0, 0].set_xticklabels(month_labels, rotation=0)
    axes[0, 0].legend(title='Category', fontsize=7, loc='upper right')
    
    # --- Size ---
    cross = pd.crosstab(df_period['month_num'], df_period['size'], normalize='index') * 100
    cross = cross.reindex(month_labels, fill_value=0)
    size_order = ['XS', 'S', 'M', 'L', 'XL', 'XXL']
    available = [s for s in size_order if s in cross.columns]
    others = [s for s in cross.columns if s not in size_order]
    cross = cross[available + others]
    cross.plot.bar(stacked=True, ax=axes[0, 1], colormap='tab10', edgecolor='white', linewidth=0.5)
    axes[0, 1].set_title(f'Size theo Thang', fontweight='bold')
    axes[0, 1].set_ylabel('%')
    axes[0, 1].set_xticklabels(month_labels, rotation=0)
    axes[0, 1].legend(title='Size', fontsize=7, loc='upper right')
    
    # --- Color ---
    cross = pd.crosstab(df_period['month_num'], df_period['color'], normalize='index') * 100
    cross = cross.reindex(month_labels, fill_value=0)
    cross.plot.bar(stacked=True, ax=axes[1, 0], colormap='tab20', edgecolor='white', linewidth=0.5)
    axes[1, 0].set_title(f'Mau sac theo Thang', fontweight='bold')
    axes[1, 0].set_ylabel('%')
    axes[1, 0].set_xticklabels(month_labels, rotation=0)
    axes[1, 0].legend(title='Color', fontsize=7, loc='upper right')
    
    # --- Segment ---
    cross = pd.crosstab(df_period['month_num'], df_period['segment'], normalize='index') * 100
    cross = cross.reindex(month_labels, fill_value=0)
    cross.plot.bar(stacked=True, ax=axes[1, 1], colormap='Pastel1', edgecolor='white', linewidth=0.5)
    axes[1, 1].set_title(f'Segment theo Thang', fontweight='bold')
    axes[1, 1].set_ylabel('%')
    axes[1, 1].set_xticklabels(month_labels, rotation=0)
    axes[1, 1].legend(title='Segment', fontsize=7, loc='upper right')
    
    plt.suptitle(f'GIAI DOAN {period_name}: Cot chong 100% theo Thang (1-12)',
                 fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

# Ve cho tung giai doan
for period in period_order:
    df_p = df[df['period'] == period]
    if len(df_p) > 0:
        plot_period_monthly(df_p, period)
    else:
        print(f'Khong co du lieu cho giai doan {period}')

## 8.3 Scatter Plot: Sum Revenue vs Sum Quantity theo San pham
San pham nao ban nhieu nhung doanh thu thap? San pham nao it ma gia cao?

In [ ]:
# 8.3 Scatter Plot: Revenue vs Quantity by Product
prod_agg = df.groupby(['product_name', 'category', 'segment']).agg(
    sum_revenue=('total_item_revenue', 'sum'),
    sum_quantity=('total_quantity', 'sum'),
    avg_price=('price', 'mean')
).reset_index()

fig = px.scatter(
    prod_agg,
    x='sum_quantity', y='sum_revenue',
    color='category', symbol='segment',
    size='avg_price', size_max=25,
    hover_name='product_name',
    hover_data={'sum_revenue': ':,.0f', 'sum_quantity': ':,.0f', 'avg_price': ':,.0f'},
    labels={
        'sum_quantity': 'Tong So luong ban (Total Quantity)',
        'sum_revenue': 'Tong Doanh thu (Total Revenue)',
        'avg_price': 'Gia TB',
        'category': 'Category',
        'segment': 'Segment'
    },
    title='Scatter: Tong Doanh thu vs Tong So luong ban theo San pham<br>'
          '<sub>Goc duoi-phai = Ban nhieu, doanh thu thap (gia re) | Goc tren-trai = Ban it, doanh thu cao (gia cao)</sub>',
    opacity=0.7
)

# Add quadrant lines at median
med_qty = prod_agg['sum_quantity'].median()
med_rev = prod_agg['sum_revenue'].median()
fig.add_hline(y=med_rev, line_dash='dash', line_color='gray', opacity=0.5,
              annotation_text=f'Median Revenue: {med_rev:,.0f}')
fig.add_vline(x=med_qty, line_dash='dash', line_color='gray', opacity=0.5,
              annotation_text=f'Median Qty: {med_qty:,.0f}')

fig.update_layout(height=650, width=1000)
fig.show()

# Static version with annotations for top outliers
fig2, ax = plt.subplots(figsize=(14, 9))

categories = prod_agg['category'].unique()
colors = sns.color_palette('husl', len(categories))
for cat, color in zip(categories, colors):
    mask = prod_agg['category'] == cat
    ax.scatter(prod_agg.loc[mask, 'sum_quantity'], prod_agg.loc[mask, 'sum_revenue'],
               alpha=0.6, s=prod_agg.loc[mask, 'avg_price'] / prod_agg['avg_price'].max() * 200 + 10,
               label=cat, color=color, edgecolor='white', linewidth=0.3)

# Annotate extreme products
top_rev = prod_agg.nlargest(5, 'sum_revenue')
top_qty = prod_agg.nlargest(5, 'sum_quantity')
outliers = pd.concat([top_rev, top_qty]).drop_duplicates('product_name')
for _, row in outliers.iterrows():
    name_short = row['product_name'][:15]
    ax.annotate(name_short, (row['sum_quantity'], row['sum_revenue']),
                fontsize=7, alpha=0.8, ha='left',
                xytext=(5, 5), textcoords='offset points')

ax.axhline(med_rev, color='gray', linestyle='--', alpha=0.4)
ax.axvline(med_qty, color='gray', linestyle='--', alpha=0.4)

# Quadrant labels
ax.text(0.02, 0.98, 'It ban, Doanh thu cao\n(Gia cao / Niche)', transform=ax.transAxes,
        fontsize=9, va='top', ha='left', color='darkred', fontstyle='italic')
ax.text(0.98, 0.02, 'Ban nhieu, Doanh thu thap\n(Gia re / Volume)', transform=ax.transAxes,
        fontsize=9, va='bottom', ha='right', color='darkblue', fontstyle='italic')
ax.text(0.98, 0.98, 'Ban nhieu, Doanh thu cao\n(Ngoi sao)', transform=ax.transAxes,
        fontsize=9, va='top', ha='right', color='darkgreen', fontstyle='italic')
ax.text(0.02, 0.02, 'It ban, Doanh thu thap\n(Can xem xet)', transform=ax.transAxes,
        fontsize=9, va='bottom', ha='left', color='gray', fontstyle='italic')

ax.set_title('Scatter: Tong Doanh thu vs Tong So luong ban theo San pham', fontweight='bold')
ax.set_xlabel('Tong So luong ban (Total Quantity)')
ax.set_ylabel('Tong Doanh thu (Total Revenue)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))
ax.legend(title='Category', fontsize=8)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()